# ProtMind — Data Collection Pipeline (Kaggle)

**What this notebook collects:**
- All reviewed human proteins from UniProt Swiss-Prot (~20,500)
- Sequences, GO annotations split by aspect, disease links, AlphaFold/PDB references
- Filters to experimental GO evidence only
- Builds the Model 1 training dataset with multi-label GO Molecular Function classes

---

## ⚠️ Kaggle-Specific Rules

| Rule | Detail |
|------|--------|
| **Session limit** | 12 hours max — run everything in one session |
| **Output folder** | `/kaggle/working/` — files here survive the session as downloadable outputs |
| **Persistence** | After the session ends, go to your notebook → **Output** tab → Download files OR save as a Kaggle Dataset |
| **Internet** | Must be enabled: **Settings → Internet → On** |
| **GPU** | Not needed for this notebook — CPU is fine |

---

## Saving Your Data After the Session
When the notebook finishes, go to **Output** tab on the right → Download `ProtMind/` folder.
Or click **Save Version** → Kaggle will save all output files permanently to your account.

---

| Cell | What it does |
|------|--------------|
| 1 | Setup folders + paths |
| 2 | Install dependencies |
| 3 | Test API connection |
| 4 | Define fetch function |
| 5 | Run collection loop |
| 6 | Inspect raw data |
| 7 | Clean data |
| 8 | Fetch GO split by aspect |
| 9 | Merge GO split |
| 10 | Build Model 1 training dataset |
| 11 | Final audit report |

In [3]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 1 — SETUP FOLDERS & PATHS
# /kaggle/working/ is Kaggle's output directory.
# Everything saved here is downloadable after the session.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

import os

# Kaggle output directory — files here are saved as notebook outputs
BASE          = "/kaggle/working/ProtMind"
RAW_DIR       = f"{BASE}/data/raw"
PROCESSED_DIR = f"{BASE}/data/processed"
LOGS_DIR      = f"{BASE}/data/logs"

for folder in [RAW_DIR, PROCESSED_DIR, LOGS_DIR]:
    os.makedirs(folder, exist_ok=True)

# All file paths used throughout the notebook
RAW_FILE      = f"{RAW_DIR}/uniprot_human_raw.tsv"
PROGRESS_FILE = f"{RAW_DIR}/collection_progress.json"
CLEAN_FILE    = f"{PROCESSED_DIR}/proteins_clean.csv"
MF_FILE       = f"{PROCESSED_DIR}/proteins_with_go_mf.csv"
VOCAB_FILE    = f"{PROCESSED_DIR}/go_mf_vocabulary.json"
AUDIT_FILE    = f"{LOGS_DIR}/collection_audit.json"
GO_PROGRESS   = f"{LOGS_DIR}/go_split_progress.json"

print("✅ Kaggle environment ready")
print(f"\n  Output folder:  {BASE}")
print(f"  Raw data:       {RAW_FILE}")
print(f"  Clean data:     {CLEAN_FILE}")
print(f"  MF dataset:     {MF_FILE}")
print(f"  GO vocab:       {VOCAB_FILE}")
print(f"\n  ⚠️  When done: Notebook → Output tab → Download or Save Version")

✅ Kaggle environment ready

  Output folder:  /kaggle/working/ProtMind
  Raw data:       /kaggle/working/ProtMind/data/raw/uniprot_human_raw.tsv
  Clean data:     /kaggle/working/ProtMind/data/processed/proteins_clean.csv
  MF dataset:     /kaggle/working/ProtMind/data/processed/proteins_with_go_mf.csv
  GO vocab:       /kaggle/working/ProtMind/data/processed/go_mf_vocabulary.json

  ⚠️  When done: Notebook → Output tab → Download or Save Version


In [4]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2 — INSTALL DEPENDENCIES
# Most are pre-installed on Kaggle. tqdm needs a refresh.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

!pip install requests pandas tqdm -q

import requests
import pandas as pd
import json
import time
import re
import os
from datetime import datetime

# Confirm internet is enabled
print("Checking internet access...")
try:
    r = requests.get("https://rest.uniprot.org", timeout=5)
    print("✅ Internet access confirmed")
except Exception:
    print("❌ No internet access!")
    print("   → Go to Settings (right panel) → Internet → Turn ON → Save")
    raise Exception("Enable internet in Kaggle settings before running this notebook")

print(f"\n  pandas   {pd.__version__}")
print(f"  requests {requests.__version__}")

Checking internet access...
✅ Internet access confirmed

  pandas   3.0.2
  requests 2.33.1


In [14]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 3 — CONNECTION TEST
# Confirms query syntax, fields, and pagination header work.
# Fix any issues here before running the full collection.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def test_uniprot_connection():
    url = "https://rest.uniprot.org/uniprotkb/search"
    params = {
        "query": "reviewed:true AND organism_id:9606 AND length:[50 TO 1000]",
        "format": "tsv",
        "fields": ",".join([
            "accession", "id", "protein_name", "gene_names",
            "organism_name", "organism_id", "sequence", "length",
            "go", "xref_pdb", "xref_alphafolddb", "annotation_score",
            "cc_disease", "ft_act_site", "ft_binding",
            "cc_function", "keyword",
        ]),
        "size": 3,
    }
    print("Testing UniProt API...")
    try:
        resp = requests.get(url, params=params, timeout=30)
        if resp.status_code != 200:
            print(f"❌ HTTP {resp.status_code}: {resp.text[:300]}")
            return False
        lines = resp.text.strip().split("\n")
        print(f"✅ API working!")
        print(f"  Status:          {resp.status_code}")
        print(f"  Proteins in test: {len(lines)-1}")
        print(f"  Cursor header:   {'Link' in resp.headers}")
        print(f"\n  Columns returned ({len(lines[0].split(chr(9)))} total):")
        for col in lines[0].split("\t"):
            print(f"    · {col}")
        print(f"\n  Sample: {lines[1].split(chr(9))[0]} — {lines[1].split(chr(9))[1]}")
        return True
    except Exception as e:
        print(f"❌ Failed: {e}")
        return False

ok = test_uniprot_connection()
if not ok:
    raise Exception("Fix API connection before proceeding")

Testing UniProt API...
✅ API working!
  Status:          200
  Proteins in test: 3
  Cursor header:   True

  Columns returned (17 total):
    · Entry
    · Entry Name
    · Protein names
    · Gene Names
    · Organism
    · Organism (ID)
    · Sequence
    · Length
    · Gene Ontology (GO)
    · PDB
    · AlphaFoldDB
    · Annotation
    · Involvement in disease
    · Active site
    · Binding site
    · Function [CC]
    · Keywords

  Sample: A0A1B0GTW7 — CIROP_HUMAN


In [16]:
def fetch_page(next_url=None, size=500):
    """
    Fetch one page of UniProt results.
    """
    if next_url is None:
        url = "https://rest.uniprot.org/uniprotkb/search"
        params = {
            "query": "reviewed:true AND organism_id:9606 AND length:[50 TO 1000]",
            "format": "tsv",
            "fields": ",".join([
                "accession", "id", "protein_name", "gene_names",
                "organism_name", "organism_id", "sequence", "length",
                "go", "xref_pdb", "xref_alphafolddb", "annotation_score",
                "cc_disease", "ft_act_site", "ft_binding",
                "cc_function", "keyword",
            ]),
            "size": size,
        }
    else:
        url    = next_url
        params = None

    for attempt in range(1, 4):
        try:
            resp = requests.get(url, params=params, timeout=45)

            if resp.status_code == 200:
                # FIX: Use regex to find the 'next' link correctly.
                # This avoids splitting on commas inside the URL's fields.
                next_link = None
                if "Link" in resp.headers:
                    import re
                    match = re.search(r'<([^>]+)>;\s*rel="next"', resp.headers["Link"])
                    if match:
                        next_link = match.group(1)
                return resp.text, next_link

            elif resp.status_code == 429:
                wait = 30 * attempt
                print(f"  ⚠️  Rate limited — waiting {wait}s...")
                time.sleep(wait)
            else:
                print(f"  HTTP {resp.status_code} (attempt {attempt}/3)")
                time.sleep(5 * attempt)

        except Exception as e:
            print(f"  Error: {e} (attempt {attempt}/3)")
            time.sleep(10 * attempt)

    return None, None


In [ ]:
# ── Load progress if resuming ──
if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE) as f:
        prog = json.load(f)
    next_url        = prog["next_url"]
    # Safety: If next_url is garbage from the previous crash, reset it
    if next_url and not next_url.startswith("http"):
        print("⚠️  Invalid cursor found in progress. Starting fresh to ensure data integrity.")
        next_url = None
        total_collected = 0
        pages_done = 0
    else:
        total_collected = prog["total_collected"]
        pages_done      = prog["pages_done"]
        print(f"▶  Resuming — {total_collected} proteins collected ({pages_done} pages)")
else:
    next_url, total_collected, pages_done = None, 0, 0
    if os.path.exists(RAW_FILE): os.remove(RAW_FILE)
    print("▶  Starting fresh collection")

start_time = time.time()
header_written = (total_collected > 0)

while True:
    page_start = time.time()
    batch_text, next_url = fetch_page(next_url=next_url, size=500)

    if batch_text is None:
        print(f"\n❌ Fetch failed. Re-run this cell to resume.")
        break

    lines = batch_text.strip().split("\n")
    if not lines or (len(lines) == 1 and lines[0].startswith("Entry")):
        print(f"\n✅ No more data. Done at {total_collected} proteins.")
        break

    # FIX: Robust header handling to prevent protein loss
    # Only drop the first line if it's the header AND we already have one.
    if lines[0].startswith("Entry"):
        data_lines = lines[1:] if header_written else lines
        header_written = True
    else:
        data_lines = lines # No header on this page (standard for cursor pages)

    actual_data = [l for l in data_lines if l.strip()]
    
    if actual_data:
        with open(RAW_FILE, "a", encoding="utf-8") as f:
            f.write("\n".join(actual_data) + "\n")

        count = len([l for l in actual_data if not l.startswith("Entry")])
        total_collected += count
        pages_done      += 1

    # Save progress
    with open(PROGRESS_FILE, "w") as f:
        json.dump({"next_url": next_url, "total_collected": total_collected,
                   "pages_done": pages_done, "last_update": datetime.now().isoformat()}, f, indent=2)

    print(f"  Page {pages_done:>3}  |  +{len(actual_data):>3}  |  Total: {total_collected:>6}  |  "
          f"{time.time()-page_start:.1f}s  |  Elapsed: {(time.time()-start_time)/60:.1f}min")

    if next_url is None:
        print(f"\n✅ All pages exhausted. Total: {total_collected} proteins")
        break
    time.sleep(0.5)


▶  Starting fresh collection
   Query: reviewed human proteins, length 50-1000 AA
   Expected: ~20,500 proteins across ~42 pages

  Page   1  |  +500  |  Total:    500  |  2.5s  |  Elapsed: 0.0min
  Error: Invalid URL 'keyword&query=reviewed%3Atrue%20AND%20organism_id%3A9606%20AND%20length%3A%5B50%20TO%201000%5D&cursor=c9bacmxsqhkqgdwwc4w9o0o7l2qlcgy6n3ynk&size=500': No scheme supplied. Perhaps you meant https://keyword&query=reviewed%3Atrue%20AND%20organism_id%3A9606%20AND%20length%3A%5B50%20TO%201000%5D&cursor=c9bacmxsqhkqgdwwc4w9o0o7l2qlcgy6n3ynk&size=500? (attempt 1/3)
  Error: Invalid URL 'keyword&query=reviewed%3Atrue%20AND%20organism_id%3A9606%20AND%20length%3A%5B50%20TO%201000%5D&cursor=c9bacmxsqhkqgdwwc4w9o0o7l2qlcgy6n3ynk&size=500': No scheme supplied. Perhaps you meant https://keyword&query=reviewed%3Atrue%20AND%20organism_id%3A9606%20AND%20length%3A%5B50%20TO%201000%5D&cursor=c9bacmxsqhkqgdwwc4w9o0o7l2qlcgy6n3ynk&size=500? (attempt 2/3)
  Error: Invalid URL 'keyword&query=r

In [8]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 6 — LOAD & INSPECT RAW DATA
# Run after collection to confirm what was captured.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

df_raw = pd.read_csv(RAW_FILE, sep="\t", on_bad_lines="skip", low_memory=False)

print(f"{'═'*55}")
print(f"  RAW DATA INSPECTION")
print(f"{'═'*55}")
print(f"  Rows:     {len(df_raw):,}")
print(f"  Columns:  {len(df_raw.columns)}")
print(f"\n  {'#':<4}  {'Column':<42}  {'Missing':>8}  {'%':>6}")
print(f"  {'─'*65}")
for i, col in enumerate(df_raw.columns):
    missing = df_raw[col].isna().sum()
    pct     = missing / len(df_raw) * 100
    flag    = " ⚠️" if pct > 30 else ""
    print(f"  [{i:>2}]  {col:<42}  {missing:>8,}  {pct:>5.1f}%{flag}")

print(f"\n  Sample row (first protein):")
for col in df_raw.columns:
    val = str(df_raw.iloc[0][col])[:80]
    print(f"    {col:<30}: {val}")

═══════════════════════════════════════════════════════
  RAW DATA INSPECTION
═══════════════════════════════════════════════════════
  Rows:     500
  Columns:  17

  #     Column                                       Missing       %
  ─────────────────────────────────────────────────────────────────
  [ 0]  Entry                                              0    0.0%
  [ 1]  Entry Name                                         0    0.0%
  [ 2]  Protein names                                      0    0.0%
  [ 3]  Gene Names                                         0    0.0%
  [ 4]  Organism                                           0    0.0%
  [ 5]  Organism (ID)                                      0    0.0%
  [ 6]  Sequence                                           0    0.0%
  [ 7]  Length                                             0    0.0%
  [ 8]  Gene Ontology (GO)                                 2    0.4%
  [ 9]  PDB                                              185   37.0% ⚠️
  [1

In [9]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 7 — CLEAN THE RAW DATA
# 7 steps: drops bad rows, renames columns, adds flags.
# Saves proteins_clean.csv to /kaggle/working/
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def find_col(df, keywords):
    """Find column by keyword — case insensitive."""
    for col in df.columns:
        for kw in keywords:
            if kw.lower() in col.lower():
                return col
    return None

# Auto-detect column names regardless of exact UniProt naming
COL_ID      = find_col(df_raw, ["entry", "accession"])
COL_NAME    = find_col(df_raw, ["entry name"])
COL_PROTEIN = find_col(df_raw, ["protein name"])
COL_GENE    = find_col(df_raw, ["gene name"])
COL_ORG     = find_col(df_raw, ["organism"])
COL_SEQ     = find_col(df_raw, ["sequence"])
COL_LEN     = find_col(df_raw, ["length"])
COL_GO      = find_col(df_raw, ["gene ontology", "go"])
COL_PDB     = find_col(df_raw, ["pdb"])
COL_AF      = find_col(df_raw, ["alphafold"])
COL_SCORE   = find_col(df_raw, ["annotation"])
COL_DISEASE = find_col(df_raw, ["disease"])
COL_FUNC    = find_col(df_raw, ["function"])
COL_ACT     = find_col(df_raw, ["active site", "act_site"])
COL_BIND    = find_col(df_raw, ["binding"])
COL_KW      = find_col(df_raw, ["keyword"])

print("Detected columns:")
detected = {
    "UniProt ID": COL_ID, "Sequence": COL_SEQ, "GO": COL_GO,
    "AlphaFold": COL_AF, "PDB": COL_PDB, "Disease": COL_DISEASE,
    "Score": COL_SCORE, "Function": COL_FUNC
}
for k, v in detected.items():
    flag = "✅" if v else "❌"
    print(f"  {flag}  {k:<15}: {v}")

df  = df_raw.copy()

# ── Step 1: Drop missing sequences ──────────────────────
before = len(df)
df     = df.dropna(subset=[COL_SEQ])
df     = df[df[COL_SEQ].str.strip() != ""]
print(f"\nStep 1 — Drop missing sequences:     {len(df):>6}  (-{before-len(df)})")

# ── Step 2: Drop duplicate UniProt IDs ──────────────────
before = len(df)
df     = df.drop_duplicates(subset=[COL_ID], keep="first")
print(f"Step 2 — Drop duplicate UniProt IDs: {len(df):>6}  (-{before-len(df)})")

# ── Step 3: Drop duplicate sequences (isoforms) ─────────
before = len(df)
df     = df.drop_duplicates(subset=[COL_SEQ], keep="first")
print(f"Step 3 — Drop duplicate sequences:   {len(df):>6}  (-{before-len(df)})")

# ── Step 4: Drop annotation score 1 ─────────────────────
if COL_SCORE:
    before        = len(df)
    df[COL_SCORE] = pd.to_numeric(df[COL_SCORE], errors="coerce")
    df            = df[df[COL_SCORE] >= 2]
    print(f"Step 4 — Drop annotation score 1:    {len(df):>6}  (-{before-len(df)})")

# ── Step 5: Validate amino acid characters ───────────────
VALID_AA     = set("ACDEFGHIKLMNPQRSTVWYX")
before       = len(df)
df["_valid"] = df[COL_SEQ].apply(
    lambda s: all(c in VALID_AA for c in str(s).upper().strip())
)
df = df[df["_valid"]].drop(columns=["_valid"])
print(f"Step 5 — Drop invalid sequences:     {len(df):>6}  (-{before-len(df)})")

# ── Step 6: Rename to clean internal names ───────────────
rename_map = {}
for old, new in [
    (COL_ID, "uniprot_id"),     (COL_NAME, "entry_name"),
    (COL_PROTEIN, "protein_name"), (COL_GENE, "gene_names"),
    (COL_ORG, "organism"),      (COL_SEQ, "sequence"),
    (COL_LEN, "length"),        (COL_GO, "go_all"),
    (COL_PDB, "pdb_ids"),       (COL_AF, "alphafold_id"),
    (COL_SCORE, "annotation_score"), (COL_DISEASE, "disease"),
    (COL_FUNC, "function_description"), (COL_ACT, "active_sites"),
    (COL_BIND, "binding_sites"), (COL_KW, "keywords"),
]:
    if old:
        rename_map[old] = new
df = df.rename(columns=rename_map)

# ── Step 7: Add derived flag columns ─────────────────────
df["seq_length"]    = df["sequence"].str.len()
df["has_pdb"]       = df["pdb_ids"].notna()
df["has_alphafold"] = df["alphafold_id"].notna()
df["has_disease"]   = df["disease"].notna()
df["has_go"]        = df["go_all"].notna() & (df["go_all"].str.strip() != "")

df.to_csv(CLEAN_FILE, index=False)

print(f"\n{'═'*50}")
print(f"  CLEANING COMPLETE")
print(f"{'═'*50}")
print(f"  Final proteins:       {len(df):>6}")
print(f"  Has AlphaFold:        {df['has_alphafold'].sum():>6}  ({df['has_alphafold'].mean()*100:.1f}%)")
print(f"  Has PDB:              {df['has_pdb'].sum():>6}  ({df['has_pdb'].mean()*100:.1f}%)")
print(f"  Has disease link:     {df['has_disease'].sum():>6}  ({df['has_disease'].mean()*100:.1f}%)")
print(f"  Has GO annotation:    {df['has_go'].sum():>6}  ({df['has_go'].mean()*100:.1f}%)")
print(f"  Saved: {CLEAN_FILE}")
print(f"{'═'*50}")

Detected columns:
  ✅  UniProt ID     : Entry
  ✅  Sequence       : Sequence
  ✅  GO             : Gene Ontology (GO)
  ✅  AlphaFold      : AlphaFoldDB
  ✅  PDB            : PDB
  ✅  Disease        : Involvement in disease
  ✅  Score          : Annotation
  ✅  Function       : Function [CC]

Step 1 — Drop missing sequences:        500  (-0)
Step 2 — Drop duplicate UniProt IDs:    500  (-0)
Step 3 — Drop duplicate sequences:      499  (-1)
Step 4 — Drop annotation score 1:       499  (-0)
Step 5 — Drop invalid sequences:        499  (-0)

══════════════════════════════════════════════════
  CLEANING COMPLETE
══════════════════════════════════════════════════
  Final proteins:          499
  Has AlphaFold:           499  (100.0%)
  Has PDB:                 315  (63.1%)
  Has disease link:        195  (39.1%)
  Has GO annotation:       497  (99.6%)
  Saved: /kaggle/working/ProtMind/data/processed/proteins_clean.csv
══════════════════════════════════════════════════


In [10]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 8 — FETCH GO ANNOTATIONS SPLIT BY ASPECT
#
# The 'go' field mixes Molecular Function + Biological
# Process + Cellular Component together.
#
# We re-fetch using go_f / go_p / go_c fields separately.
# Model 1 trains ONLY on Molecular Function (go_f).
#
# Saves progress every 5 batches — resume-safe.
# Expected: ~5-10 minutes
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def fetch_go_split_batch(uniprot_ids):
    """Fetch GO split into MF / BP / CC for a list of IDs."""
    id_str = " OR ".join([f"accession:{uid}" for uid in uniprot_ids])
    url    = "https://rest.uniprot.org/uniprotkb/search"
    params = {
        "query":  id_str,
        "format": "tsv",
        "fields": "accession,go_f,go_p,go_c",
        "size":   len(uniprot_ids),
    }
    for attempt in range(3):
        try:
            resp = requests.get(url, params=params, timeout=45)
            if resp.status_code == 200:
                return resp.text
            time.sleep(5 * (attempt + 1))
        except Exception:
            time.sleep(10 * (attempt + 1))
    return None

df_clean    = pd.read_csv(CLEAN_FILE, low_memory=False)
all_ids     = df_clean["uniprot_id"].tolist()
total       = len(all_ids)

# Resume if cell was interrupted this session
go_records  = {}
done_so_far = set()
if os.path.exists(GO_PROGRESS):
    with open(GO_PROGRESS) as f:
        saved = json.load(f)
    go_records  = saved.get("records", {})
    done_so_far = set(saved.get("done_ids", []))
    print(f"Resuming — {len(done_so_far):,} already fetched")

remaining  = [uid for uid in all_ids if uid not in done_so_far]
BATCH_SIZE = 200
batches    = [remaining[i:i+BATCH_SIZE]
               for i in range(0, len(remaining), BATCH_SIZE)]
failed_ids = []

print(f"Fetching GO split for {len(remaining):,} proteins "
      f"in {len(batches)} batches...")
print()

for batch_num, batch in enumerate(batches):
    result = fetch_go_split_batch(batch)

    if result is None:
        failed_ids.extend(batch)
        print(f"  Batch {batch_num+1}/{len(batches)} ❌ failed")
        continue

    for line in result.strip().split("\n")[1:]:
        parts = line.split("\t")
        if len(parts) >= 1:
            uid = parts[0].strip()
            go_records[uid] = {
                "go_molecular_function": parts[1].strip() if len(parts) > 1 else "",
                "go_biological_process": parts[2].strip() if len(parts) > 2 else "",
                "go_cellular_component": parts[3].strip() if len(parts) > 3 else "",
            }
            done_so_far.add(uid)

    # Save progress every 5 batches
    if batch_num % 5 == 0:
        with open(GO_PROGRESS, "w") as f:
            json.dump({"records": go_records,
                       "done_ids": list(done_so_far)}, f)

    if batch_num % 10 == 0 or batch_num == len(batches) - 1:
        print(f"  Batch {batch_num+1:>3}/{len(batches)}  ✓  "
              f"{len(go_records):>6,}/{total:>6,} proteins")

    time.sleep(0.3)

# Final save
with open(GO_PROGRESS, "w") as f:
    json.dump({"records": go_records,
               "done_ids": list(done_so_far)}, f)

print(f"\n✅ GO split complete")
print(f"  Proteins fetched: {len(go_records):,}")
print(f"  Failed batches:   {len(failed_ids)}")

Fetching GO split for 499 proteins in 3 batches...

  Batch 1/3 ❌ failed
  Batch 2/3 ❌ failed
  Batch   3/3  ✓      99/   499 proteins

✅ GO split complete
  Proteins fetched: 99
  Failed batches:   400


In [11]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 9 — MERGE GO SPLIT INTO MAIN DATASET
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

with open(GO_PROGRESS) as f:
    saved = json.load(f)
go_records = saved["records"]

go_df = pd.DataFrame.from_dict(go_records, orient="index")
go_df.index.name = "uniprot_id"
go_df = go_df.reset_index()

df_clean  = pd.read_csv(CLEAN_FILE, low_memory=False)
df_merged = df_clean.merge(go_df, on="uniprot_id", how="left")

df_merged["has_go_mf"] = (
    df_merged["go_molecular_function"].notna() &
    (df_merged["go_molecular_function"].str.strip() != "")
)

df_merged.to_csv(CLEAN_FILE, index=False)

print(f"{'═'*50}")
print(f"  GO SPLIT MERGED")
print(f"{'═'*50}")
print(f"  Total proteins:                {len(df_merged):>6,}")
mf_count = df_merged['has_go_mf'].sum()
print(f"  Has GO Molecular Function:     {mf_count:>6,}  ({mf_count/len(df_merged)*100:.1f}%)")
bp_count = df_merged['go_biological_process'].notna().sum()
print(f"  Has GO Biological Process:     {bp_count:>6,}")
cc_count = df_merged['go_cellular_component'].notna().sum()
print(f"  Has GO Cellular Component:     {cc_count:>6,}")
print(f"  Saved: {CLEAN_FILE}")
print(f"{'═'*50}")

══════════════════════════════════════════════════
  GO SPLIT MERGED
══════════════════════════════════════════════════
  Total proteins:                   499
  Has GO Molecular Function:         91  (18.2%)
  Has GO Biological Process:         99
  Has GO Cellular Component:         99
  Saved: /kaggle/working/ProtMind/data/processed/proteins_clean.csv
══════════════════════════════════════════════════


In [12]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 10 — BUILD MODEL 1 TRAINING DATASET
#
# Keeps only proteins with GO Molecular Function.
# Removes GO terms appearing in < MIN_TERM_COUNT proteins.
# Builds the binary label matrix for multi-label training.
# Saves proteins_with_go_mf.csv + go_mf_vocabulary.json
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

MIN_TERM_COUNT = 20   # Min proteins per GO MF term to include as a class

df_full = pd.read_csv(CLEAN_FILE, low_memory=False)
df_mf   = df_full[df_full["has_go_mf"] == True].copy()
print(f"Proteins with GO Molecular Function: {len(df_mf):,}")

def parse_go_terms(go_string):
    """
    Parse 'ATP binding [GO:0005524]; kinase activity [GO:0016301]'
    → [("ATP binding", "GO:0005524"), ("kinase activity", "GO:0016301")]
    """
    if pd.isna(go_string) or str(go_string).strip() == "":
        return []
    terms = []
    for entry in str(go_string).split(";"):
        entry = entry.strip()
        match = re.search(r'\[GO:(\d+)\]', entry)
        if match:
            go_id     = f"GO:{match.group(1)}"
            term_name = re.sub(r'\s*\[GO:\d+\].*$', '', entry).strip()
            terms.append((term_name, go_id))
    return terms

print("Parsing GO terms...")
df_mf["go_mf_parsed"] = df_mf["go_molecular_function"].apply(parse_go_terms)
df_mf["go_mf_count"]  = df_mf["go_mf_parsed"].apply(len)

# Count term frequency across dataset
all_go_ids   = []
all_go_names = {}
for terms in df_mf["go_mf_parsed"]:
    for name, go_id in terms:
        all_go_ids.append(go_id)
        all_go_names[go_id] = name
go_id_counts = pd.Series(all_go_ids).value_counts()

valid_go_ids = set(go_id_counts[go_id_counts >= MIN_TERM_COUNT].index)

print(f"\nGO MF statistics:")
print(f"  Unique GO MF terms:            {len(go_id_counts):>6,}")
print(f"  Terms with >= {MIN_TERM_COUNT} proteins:  {len(valid_go_ids):>6,}  ← label classes")
print(f"  Terms with <  {MIN_TERM_COUNT} proteins:  "
      f"{(go_id_counts < MIN_TERM_COUNT).sum():>6,}  ← excluded (too rare)")

# Keep only proteins with at least 1 valid GO MF label
df_mf["has_valid_go_mf"] = df_mf["go_mf_parsed"].apply(
    lambda terms: any(go_id in valid_go_ids for _, go_id in terms)
)
df_train = df_mf[df_mf["has_valid_go_mf"]].copy()
print(f"\nTraining-eligible proteins:        {len(df_train):>6,}")

# Build binary label matrix
sorted_go_ids = sorted(valid_go_ids)

def build_label_vector(terms):
    present = {go_id for _, go_id in terms}
    return {go_id: int(go_id in present) for go_id in sorted_go_ids}

print("Building multi-label binary matrix...")
label_df = pd.DataFrame(
    df_train["go_mf_parsed"].apply(build_label_vector).tolist(),
    index=df_train.index
)

# Select core columns + attach label columns
core_cols = [
    "uniprot_id", "entry_name", "protein_name", "gene_names",
    "sequence", "seq_length", "annotation_score",
    "has_pdb", "has_alphafold", "has_disease", "disease",
    "go_molecular_function", "go_biological_process",
    "go_cellular_component", "alphafold_id", "pdb_ids",
    "function_description",
]
core_cols      = [c for c in core_cols if c in df_train.columns]
df_train_final = pd.concat([
    df_train[core_cols].reset_index(drop=True),
    label_df.reset_index(drop=True)
], axis=1)

df_train_final.to_csv(MF_FILE, index=False)

# Save GO vocabulary (needed at inference time)
vocab = {
    go_id: {"name":  all_go_names.get(go_id, ""),
             "count": int(go_id_counts.get(go_id, 0))}
    for go_id in sorted_go_ids
}
with open(VOCAB_FILE, "w") as f:
    json.dump({"go_terms":    sorted_go_ids,
               "vocabulary":  vocab,
               "min_count":   MIN_TERM_COUNT,
               "total_terms": len(sorted_go_ids)}, f, indent=2)

print(f"\n{'═'*55}")
print(f"  MODEL 1 TRAINING DATASET READY")
print(f"{'═'*55}")
print(f"  Proteins:           {len(df_train_final):>6,}")
print(f"  GO MF label classes:{len(sorted_go_ids):>6,}")
print(f"  Columns total:      {len(df_train_final.columns):>6,}")
print(f"  Saved: {MF_FILE}")
print(f"  Vocab: {VOCAB_FILE}")
print(f"\n  Top 15 GO MF label classes:")
print(f"  {'COUNT':>6}  {'GO ID':<14}  TERM")
print(f"  {'─'*60}")
for go_id in sorted_go_ids[:15]:
    count = go_id_counts.get(go_id, 0)
    name  = all_go_names.get(go_id, "")[:45]
    print(f"  {count:>6,}  {go_id:<14}  {name}")
print(f"{'═'*55}")

Proteins with GO Molecular Function: 91
Parsing GO terms...

GO MF statistics:
  Unique GO MF terms:               302
  Terms with >= 20 proteins:       0  ← label classes
  Terms with <  20 proteins:     302  ← excluded (too rare)

Training-eligible proteins:             0
Building multi-label binary matrix...

═══════════════════════════════════════════════════════
  MODEL 1 TRAINING DATASET READY
═══════════════════════════════════════════════════════
  Proteins:                0
  GO MF label classes:     0
  Columns total:          17
  Saved: /kaggle/working/ProtMind/data/processed/proteins_with_go_mf.csv
  Vocab: /kaggle/working/ProtMind/data/processed/go_mf_vocabulary.json

  Top 15 GO MF label classes:
   COUNT  GO ID           TERM
  ────────────────────────────────────────────────────────────
═══════════════════════════════════════════════════════


In [13]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 11 — FINAL AUDIT REPORT
# Full summary of everything collected and built.
# After this cell: go to Output tab → Download files.
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

df_full  = pd.read_csv(CLEAN_FILE, low_memory=False)
df_model = pd.read_csv(MF_FILE,    low_memory=False)
label_cols = [c for c in df_model.columns if c.startswith("GO:")]

cca_kw  = ["cholangiocarcinoma", "bile duct", "biliary",
            "liver", "hepato", "cancer", "carcinoma",
            "tumor", "oncogene"]
cca_cnt = 0
if "disease" in df_full.columns:
    cca_cnt = df_full["disease"].dropna().apply(
        lambda x: any(k in str(x).lower() for k in cca_kw)
    ).sum()

print(f"\n{'═'*55}")
print(f"  PROTMIND — FINAL DATA AUDIT")
print(f"{'═'*55}")

print(f"\n  LAYER 1 — proteins_clean.csv")
print(f"  {'─'*48}")
print(f"  Total proteins:              {len(df_full):>7,}")
print(f"  All human (Homo sapiens):    ✅")
print(f"  All Swiss-Prot reviewed:     ✅")
print(f"  Duplicate IDs:               0")
print(f"  Missing sequences:           0")
af  = df_full['has_alphafold'].sum()
pdb = df_full['has_pdb'].sum()
dis = df_full['has_disease'].sum()
go  = df_full['has_go'].sum()
print(f"  Has AlphaFold:               {af:>7,}  ({af/len(df_full)*100:.1f}%)")
print(f"  Has PDB:                     {pdb:>7,}  ({pdb/len(df_full)*100:.1f}%)")
print(f"  Has disease link:            {dis:>7,}  ({dis/len(df_full)*100:.1f}%)")
print(f"  Has GO annotation:           {go:>7,}  ({go/len(df_full)*100:.1f}%)")
print(f"  CCA/cancer-relevant:         {cca_cnt:>7,}")

print(f"\n  LAYER 2 — proteins_with_go_mf.csv")
print(f"  {'─'*48}")
print(f"  Training proteins:           {len(df_model):>7,}")
print(f"  GO MF label classes:         {len(label_cols):>7,}")
if label_cols:
    label_sums    = df_model[label_cols].sum()
    mean_per_prot = df_model[label_cols].sum(axis=1).mean()
    print(f"  Avg labels per protein:      {mean_per_prot:>7.1f}")
    print(f"  Most common class:           {label_sums.idxmax()} ({label_sums.max():.0f})")
    print(f"  Least common class:          {label_sums.idxmin()} ({label_sums.min():.0f})")

print(f"\n  OUTPUT FILES  ({BASE})")
print(f"  {'─'*48}")
files = [RAW_FILE, CLEAN_FILE, MF_FILE, VOCAB_FILE, AUDIT_FILE]
for fpath in files:
    if os.path.exists(fpath):
        size = os.path.getsize(fpath) / 1024 / 1024
        print(f"  ✅  {os.path.basename(fpath):<40} {size:.1f} MB")
    else:
        print(f"  ❌  {os.path.basename(fpath):<40} not found")

# Save audit JSON
audit = {
    "completed_at":       datetime.now().isoformat(),
    "total_clean":        len(df_full),
    "total_training":     len(df_model),
    "go_mf_classes":      len(label_cols),
    "has_alphafold_pct":  round(af/len(df_full)*100, 1),
    "has_pdb_pct":        round(pdb/len(df_full)*100, 1),
    "cca_relevant":       int(cca_cnt),
}
with open(AUDIT_FILE, "w") as f:
    json.dump(audit, f, indent=2)

print(f"\n{'═'*55}")
print(f"  ✅  DATA COLLECTION PHASE COMPLETE")
print(f"{'═'*55}")
print(f"\n  ⬇️  Download files:")
print(f"     Notebook page → Output tab (right side) → Download")
print(f"     OR: Save Version → files saved to your Kaggle account")
print(f"\n  Next notebooks:")
print(f"  1. blast_search.ipynb         ← BLAST context")
print(f"  2. generate_embeddings.ipynb  ← ESM-2 embeddings (P100 GPU)")
print(f"  3. train_model1.ipynb         ← Function prediction model")


═══════════════════════════════════════════════════════
  PROTMIND — FINAL DATA AUDIT
═══════════════════════════════════════════════════════

  LAYER 1 — proteins_clean.csv
  ────────────────────────────────────────────────
  Total proteins:                  499
  All human (Homo sapiens):    ✅
  All Swiss-Prot reviewed:     ✅
  Duplicate IDs:               0
  Missing sequences:           0
  Has AlphaFold:                   499  (100.0%)
  Has PDB:                         315  (63.1%)
  Has disease link:                195  (39.1%)
  Has GO annotation:               497  (99.6%)
  CCA/cancer-relevant:              34

  LAYER 2 — proteins_with_go_mf.csv
  ────────────────────────────────────────────────
  Training proteins:                 0
  GO MF label classes:               0

  OUTPUT FILES  (/kaggle/working/ProtMind)
  ────────────────────────────────────────────────
  ✅  uniprot_human_raw.tsv                    1.5 MB
  ✅  proteins_clean.csv                       1.7 MB
  ✅ 